# London Data Preparation

This notebook prepares the London AirDNA/Airbnb dataset for the capstone project.

## Final outputs

- `london_listings_clean.csv`  
  - Unit of analysis: **1 row = 1 Airbnb listing**
- `london_monthly_metrics_clean.csv`  
  - Unit of analysis: **1 row = 1 Airbnb listing + 1 month**

The monthly dataset is extracted from the embedded JSON stored in the `months` column of the listings dataset.


## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

## 2. Define project paths

The notebook assumes it is executed from the `notebooks/` folder.

Expected structure:

```text
KPMG_Airbnb_Capstone/
├── data/
│   ├── raw/
│   │   └── london/
│   └── processed/
│       └── london/
└── notebooks/
```


In [2]:
BASE_DIR = Path("..")

RAW_DIR = BASE_DIR / "data" / "raw" / "london"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "london"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw directory:", RAW_DIR.resolve())
print("Processed directory:", PROCESSED_DIR.resolve())

Raw directory: /Users/bobbypakenham/KPMG_Airbnb_Capstone/data/raw/london
Processed directory: /Users/bobbypakenham/KPMG_Airbnb_Capstone/data/processed/london


## 3. Load raw London listings dataset

This notebook uses the listings dataset as the main raw source because it contains:

- one row per listing,
- listing and host characteristics,
- location variables,
- performance metrics,
- the embedded monthly history in the `months` column.

If your local file has `(1)` in the name, rename it to:

```text
listings_LONDON_CONVERT_FROM_PARQUET.csv
```


In [3]:
listings_file = RAW_DIR / "listings_LONDON_CONVERT_FROM_PARQUET.csv"

if not listings_file.exists():
    raise FileNotFoundError(
        f"File not found: {listings_file}\n"
        "Please check that the file exists in data/raw/london/ "
        "and rename it to listings_LONDON_CONVERT_FROM_PARQUET.csv"
    )

listings = pd.read_csv(listings_file)

print("Listings shape:", listings.shape)

Listings shape: (9643, 81)


In [4]:
listings.head()

,listing_id,host_id,instant_book,professional_management,county,latitude,longitude,guests,bedrooms,listing_type,...,l90d_bookable_days,l90d_occupancy,l90d_adjusted_occupancy,l90d_revpar,l90d_adjusted_revpar,l90d_native_revpar,l90d_native_adjusted_revpar,l90d_avg_rate,l90d_avg_native_rate,months
0,44995702,b07504e62598,NaN,False,Greater London,5150363000,-26812000,6.0,3.0,Entire condo,...,90.0,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,"[{""date"": 1614556800000, ""available_days"": 31,..."
1,810126,e3aff5b9384a,False,False,Greater London,5151065000,-27588000,2.0,1.0,Entire rental unit,...,0.0,0.000,0.000,0.0,0.0,0.0,0.0,137.3,102.1,"[{""date"": 1614556800000, ""available_days"": 31,..."
2,51965328,d04fdbb23d97,NaN,False,Greater London,5146650000,-12650000,NaN,NaN,Private room in home,...,90.0,0.422,0.422,60.8,60.8,45.2,45.2,133.1,98.9,"[{""date"": 1614556800000, ""available_days"": 0, ..."
3,15788028,37f0ef6ed54a,NaN,NaN,Greater London,5148039000,-11858000,3.0,1.0,Entire rental unit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[{""date"": 1614556800000, ""available_days"": 31,..."
4,1095577479826242355,cb31d50d19e6,NaN,False,Greater London,5153340000,-13680000,6.0,3.0,Private room in rental unit,...,90.0,0.900,0.900,237.7,237.7,176.7,176.7,261.8,194.7,"[{""date"": 1717200000000, ""available_days"": 30,..."


## 4. Initial data audit

Before cleaning, we confirm the unit of analysis and inspect the main structure of the dataset.


In [5]:
print("Rows:", len(listings))
print("Unique listing_id:", listings["listing_id"].nunique())
print("Duplicate listing_id:", listings["listing_id"].duplicated().sum())

Rows: 9643
Unique listing_id: 9643
Duplicate listing_id: 0


Expected interpretation:

- If `Rows` equals `Unique listing_id`
- And `Duplicate listing_id` equals `0`

then **1 row = 1 unique Airbnb listing**.


In [6]:
listings.dtypes.value_counts()

float64    57
str        14
object      6
int64       4
Name: count, dtype: int64

In [7]:
missing = (
    listings.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing.head(30)

instant_book                       8238
professional_management            5265
native_extra_guest_fee             4851
extra_guest_fee                    4851
native_cleaning_fee                3979
cleaning_fee                       3979
registration                       3860
cohost                             3860
l90d_available_days                2696
l90d_blocked_days                  2696
l90d_bookable_days                 2696
ttm_days_booked                    2696
ttm_total_days                     2696
ttm_blocked_days                   2696
ttm_available_days                 2696
ttm_unavailable_days               2696
ttm_revenue                        2696
ttm_native_revenue                 2696
ttm_avg_rate                       2696
ttm_avg_native_rate                2696
ttm_months_with_data               2696
l90d_reservations_count            2696
l90d_native_revenue                2696
ttm_reservations_count             2696
ttm_cleaning_fee_revenue           2696


In [8]:
missing_pct = (
    listings.isnull()
    .mean()
    .sort_values(ascending=False)
    * 100
)

missing_pct.head(30)

instant_book                       85.429845
professional_management            54.599191
native_extra_guest_fee             50.305921
extra_guest_fee                    50.305921
native_cleaning_fee                41.263092
cleaning_fee                       41.263092
registration                       40.029037
cohost                             40.029037
l90d_available_days                27.958104
l90d_blocked_days                  27.958104
l90d_bookable_days                 27.958104
ttm_days_booked                    27.958104
ttm_total_days                     27.958104
ttm_blocked_days                   27.958104
ttm_available_days                 27.958104
ttm_unavailable_days               27.958104
ttm_revenue                        27.958104
ttm_native_revenue                 27.958104
ttm_avg_rate                       27.958104
ttm_avg_native_rate                27.958104
ttm_months_with_data               27.958104
l90d_reservations_count            27.958104
l90d_nativ

In [9]:
column_audit = pd.DataFrame({
    "column": listings.columns,
    "dtype": listings.dtypes.values,
    "missing_pct": listings.isnull().mean().values * 100
}).sort_values("missing_pct", ascending=False)

column_audit.head(30)

,column,dtype,missing_pct
2,instant_book,object,85.429845
3,professional_management,object,54.599191
39,native_extra_guest_fee,float64,50.305921
25,extra_guest_fee,float64,50.305921
38,native_cleaning_fee,float64,41.263092
24,cleaning_fee,float64,41.263092
20,registration,object,40.029037
19,cohost,object,40.029037
64,l90d_available_days,float64,27.958104
63,l90d_blocked_days,float64,27.958104


## 5. Geographic Variables

For London:

- `neighborhood` corresponds to the broader borough level (e.g., Westminster, Camden, Southwark, Tower Hamlets).
- `subdivision` corresponds to the more granular neighbourhood/local area level (e.g., Fitzrovia, Soho, Shoreditch, Stockwell).

For this project, `subdivision` should be the main geographic variable for neighbourhood-level analysis, while `neighborhood` is useful for higher-level aggregation and city-wide comparisons.


In [10]:
print("Neighbourhood-like columns:")
print([c for c in listings.columns if "neigh" in c.lower()])

print("\nSubdivision-like columns:")
print([c for c in listings.columns if "sub" in c.lower()])

Neighbourhood-like columns:
['neighborhood']

Subdivision-like columns:
['subdivision']


In [11]:
listings["neighborhood"].value_counts(dropna=False).head(20)

neighborhood
Royal Borough of Kensington and Chelsea     787
London Borough of Camden                    770
London Borough of Tower Hamlets             768
London Borough of Hackney                   563
London Borough of Southwark                 543
London Borough of Wandsworth                511
London Borough of Lambeth                   508
London Borough of Islington                 487
London Borough of Hammersmith and Fulham    477
London Borough of Brent                     353
London Borough of Barnet                    263
London Borough of Newham                    258
London Borough of Lewisham                  248
London Borough of Haringey                  222
London Borough of Ealing                    216
Paddington                                  206
Royal Borough of Greenwich                  204
London Borough of Waltham Forest            195
Marylebone                                  178
London Borough of Hounslow                  167
Name: count, dtype: int64

In [12]:
listings["subdivision"].value_counts(dropna=False).head(20)

subdivision
NaN                    1157
Whitechapel             242
Westbourne Green        210
Earl's Court            168
Fulham                  139
West Kensington         127
Chelsea                 123
Notting Hill            119
Barnsbury               109
Fitzrovia               104
North Kensington        104
Brompton                100
Elephant and Castle      92
South Kensington         87
Shepherd's Bush          87
Clapham Junction         87
Bethnal Green            86
The Borough              85
King's Cross             85
Battersea                81
Name: count, dtype: int64

## 6. Host audit

This is useful for measuring commercialisation and host concentration.


In [13]:
print("Listings:", listings["listing_id"].nunique())
print("Unique hosts:", listings["host_id"].nunique())

host_listing_counts = (
    listings.groupby("host_id")["listing_id"]
    .nunique()
    .sort_values(ascending=False)
)

host_listing_counts.describe()

Listings: 9643
Unique hosts: 7312


count    7312.000000
mean        1.318791
std         1.516422
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        49.000000
Name: listing_id, dtype: float64

In [14]:
host_listing_counts.head(20)

host_id
c7aabdfa4900    49
e41c72bc8e56    44
0fd949405a35    38
cdad55b06647    32
9ce980aa0cd8    29
4d241f643485    25
726d1a8f3a97    23
ab7675cea372    23
6dde65fb1d90    22
352fac4b1cc9    20
b26370a7e12e    17
fac8283f8239    16
76da112ef91e    15
3827675ddb05    14
db1ac9b6b931    13
19d420452d75    13
648321130eb6    13
f4d2b1e42075    12
391ee79a045f    12
e217b8a43029    12
Name: listing_id, dtype: int64

## 7. Select core listing-level columns

The goal is to keep only columns that are useful for the capstone:

- listing identification,
- host identification,
- location,
- property characteristics,
- review/rating information,
- compliance/licensing,
- recent performance metrics,
- embedded monthly history.


In [15]:
core_columns = [
    "listing_id",
    "host_id",

    "professional_management",
    "superhost",
    "cohost",

    "neighborhood",
    "subdivision",

    "latitude",
    "longitude",

    "listing_type",
    "room_type",

    "guests",
    "bedrooms",
    "beds",
    "baths",

    "num_reviews",
    "star_rating",

    "registration",

    "ttm_revenue",
    "ttm_days_booked",
    "ttm_avg_rate",

    "l90d_revenue",
    "l90d_occupancy",
    "l90d_revpar",

    "months"
]

missing_cols = [c for c in core_columns if c not in listings.columns]
print("Missing selected columns:", missing_cols)

Missing selected columns: []


In [16]:
if missing_cols:
    raise ValueError(f"The following selected columns are missing: {missing_cols}")

listings_clean = listings[core_columns].copy()

print("Listings clean shape:", listings_clean.shape)
listings_clean.head()

Listings clean shape: (9643, 25)


,listing_id,host_id,professional_management,superhost,cohost,neighborhood,subdivision,latitude,longitude,listing_type,...,num_reviews,star_rating,registration,ttm_revenue,ttm_days_booked,ttm_avg_rate,l90d_revenue,l90d_occupancy,l90d_revpar,months
0,44995702,b07504e62598,False,False,True,London Borough of Ealing,NaN,5150363000,-26812000,Entire condo,...,6.0,500.0,False,0.0,0.0,0.0,0.0,0.000,0.0,"[{""date"": 1614556800000, ""available_days"": 31,..."
1,810126,e3aff5b9384a,False,False,False,London Borough of Ealing,NaN,5151065000,-27588000,Entire rental unit,...,66.0,451.0,False,0.0,0.0,132.1,0.0,0.000,0.0,"[{""date"": 1614556800000, ""available_days"": 31,..."
2,51965328,d04fdbb23d97,False,True,False,London Borough of Lambeth,Stockwell,5146650000,-12650000,Private room in home,...,210.0,499.0,False,26655.0,204.0,127.5,5474.0,0.422,60.8,"[{""date"": 1614556800000, ""available_days"": 0, ..."
3,15788028,37f0ef6ed54a,NaN,False,NaN,London Borough of Lambeth,Oval,5148039000,-11858000,Entire rental unit,...,33.0,482.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[{""date"": 1614556800000, ""available_days"": 31,..."
4,1095577479826242355,cb31d50d19e6,False,True,True,London Borough of Camden,Fitzrovia,5153340000,-13680000,Private room in rental unit,...,56.0,438.0,False,115676.0,342.0,324.0,22495.0,0.900,237.7,"[{""date"": 1717200000000, ""available_days"": 30,..."


## 7.5 Data-type cleanup

Fix three known issues from the raw AirDNA export before any feature engineering:

- `latitude` and `longitude` are stored as integers scaled by **1e8** — rescale to decimal degrees so maps and distance calcs work.
- `star_rating` is stored as integer scaled by **100** — rescale to the 0-5 scale users expect.
- Add a `has_subdivision` flag so downstream analysis can filter out listings that can't be located at the neighbourhood level.
- Add `professional_management_known` so downstream can distinguish a *real* False from an *imputed-from-null* False.

In [17]:
# 1. Rescale latitude / longitude from integer (x1e8) to decimal degrees
for col in ("latitude", "longitude"):
    listings_clean[col] = listings_clean[col].astype(float) / 1e8

# 2. Rescale star_rating from integer (x100) to 0-5
listings_clean["star_rating"] = listings_clean["star_rating"].astype(float) / 100

# 3. Subdivision is the main geographic key — flag rows where it is missing
listings_clean["has_subdivision"] = listings_clean["subdivision"].notna()

# 4. Track whether professional_management was actually reported (vs imputed from null)
listings_clean["professional_management_known"] = listings_clean["professional_management"].notna()

print("latitude range  :", round(listings_clean["latitude"].min(), 4),
      "to", round(listings_clean["latitude"].max(), 4))
print("longitude range :", round(listings_clean["longitude"].min(), 4),
      "to", round(listings_clean["longitude"].max(), 4))
print("star_rating range:", listings_clean["star_rating"].min(),
      "to", listings_clean["star_rating"].max())
print("subdivision present:", int(listings_clean["has_subdivision"].sum()),
      "of", len(listings_clean))
print("professional_management known:", int(listings_clean["professional_management_known"].sum()),
      "of", len(listings_clean))

latitude range  : 51.3074 to 51.673
longitude range : -0.5019 to 0.2348
star_rating range: 1.0 to 5.0
subdivision present: 8486 of 9643
professional_management known: 4378 of 9643


## 8. Create listing-level features

These features will be useful for the risk score, clustering and policy analysis.


In [18]:
# Count how many listings each host manages
host_listing_count = listings_clean.groupby("host_id")["listing_id"].transform("nunique")

listings_clean["host_listing_count"] = host_listing_count
listings_clean["multi_listing_host"] = listings_clean["host_listing_count"] > 1
listings_clean["host_5_plus_listings"] = listings_clean["host_listing_count"] >= 5
listings_clean["host_10_plus_listings"] = listings_clean["host_listing_count"] >= 10

# Entire-home flag (explicit equality on the normalised room_type)
listings_clean["entire_home_flag"] = (
    listings_clean["room_type"].astype(str).str.lower() == "entire_home"
)

# Professional management flag.
# Keep original value, but create a boolean flag where available.
listings_clean["professional_management_flag"] = (
    listings_clean["professional_management"]
    .fillna(False)
    .astype(bool)
)

# Registration / licence availability flag.
listings_clean["has_registration"] = listings_clean["registration"].notna()

listings_clean.head()

,listing_id,host_id,professional_management,superhost,cohost,neighborhood,subdivision,latitude,longitude,listing_type,...,months,has_subdivision,professional_management_known,host_listing_count,multi_listing_host,host_5_plus_listings,host_10_plus_listings,entire_home_flag,professional_management_flag,has_registration
0,44995702,b07504e62598,False,False,True,London Borough of Ealing,NaN,51.50363,-0.26812,Entire condo,...,"[{""date"": 1614556800000, ""available_days"": 31,...",False,True,1,False,False,False,True,False,True
1,810126,e3aff5b9384a,False,False,False,London Borough of Ealing,NaN,51.51065,-0.27588,Entire rental unit,...,"[{""date"": 1614556800000, ""available_days"": 31,...",False,True,2,True,False,False,True,False,True
2,51965328,d04fdbb23d97,False,True,False,London Borough of Lambeth,Stockwell,51.46650,-0.12650,Private room in home,...,"[{""date"": 1614556800000, ""available_days"": 0, ...",True,True,2,True,False,False,False,False,True
3,15788028,37f0ef6ed54a,NaN,False,NaN,London Borough of Lambeth,Oval,51.48039,-0.11858,Entire rental unit,...,"[{""date"": 1614556800000, ""available_days"": 31,...",True,False,1,False,False,False,True,False,False
4,1095577479826242355,cb31d50d19e6,False,True,True,London Borough of Camden,Fitzrovia,51.53340,-0.13680,Private room in rental unit,...,"[{""date"": 1717200000000, ""available_days"": 30,...",True,True,3,True,False,False,False,False,True


## 9. Export listing-level clean dataset

In [19]:
listings_clean.to_csv(
    PROCESSED_DIR / "london_listings_clean.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "london_listings_clean.csv")

Saved: ../data/processed/london/london_listings_clean.csv


## 10. Inspect embedded monthly history

The `months` column is a JSON string. Each row contains a list of monthly performance records for that listing.


In [20]:
sample_months = json.loads(listings.loc[0, "months"])

print("Type:", type(sample_months))
print("Number of months in first listing:", len(sample_months))
sample_months[0]

Type: <class 'list'>
Number of months in first listing: 38


{'date': 1614556800000,
 'available_days': 31,
 'unavailable_days': 0,
 'occupancy': 0,
 'rate_avg': 243.1,
 'native_rate_avg': 175,
 'revenue': 0,
 'native_revenue': 0,
 'active': False,
 'booking_lead_time_avg': None,
 'length_of_stay_avg': None,
 'booked_rate_avg': None,
 'native_booked_rate_avg': None,
 'rev_par': 0,
 'native_rev_par': 0}

In [21]:
sample_df = pd.DataFrame(sample_months)

sample_df["month_date"] = pd.to_datetime(
    sample_df["date"],
    unit="ms"
)

sample_df.head()

,date,available_days,unavailable_days,occupancy,rate_avg,native_rate_avg,revenue,native_revenue,active,booking_lead_time_avg,length_of_stay_avg,booked_rate_avg,native_booked_rate_avg,rev_par,native_rev_par,month_date
0,1614556800000,31,0,0.000,243.1,175,0,0,False,None,None,NaN,NaN,0.000000,0,2021-03-01
1,1617235200000,30,0,0.000,245.4,177,0,0,False,None,None,NaN,NaN,0.000000,0,2021-04-01
2,1619827200000,23,8,0.258,247.6,179,2003,1450,True,None,None,250.4,181.0,64.612903,47,2021-05-01
3,1622505600000,24,6,0.200,263.4,186,1537,1086,True,None,None,256.2,181.0,51.233333,36,2021-06-01
4,1625097600000,0,31,1.000,246.3,179,7635,5548,True,None,None,246.3,179.0,246.290323,179,2021-07-01


## 11. Expand monthly history for all listings

This transforms the embedded `months` JSON into a proper monthly panel dataset.

Final unit of analysis:

```text
1 row = 1 Airbnb listing + 1 month
```


In [22]:
monthly_records = []

for _, row in listings.iterrows():
    listing_id = row["listing_id"]

    if pd.isna(row["months"]):
        continue

    months_data = json.loads(row["months"])

    for month in months_data:
        month_record = month.copy()
        month_record["listing_id"] = listing_id
        monthly_records.append(month_record)

monthly_df = pd.DataFrame(monthly_records)

monthly_df["month_date"] = pd.to_datetime(
    monthly_df["date"],
    unit="ms"
)

print("Monthly dataset shape:", monthly_df.shape)
monthly_df.head()

Monthly dataset shape: (306822, 17)


,date,available_days,unavailable_days,occupancy,rate_avg,native_rate_avg,revenue,native_revenue,active,booking_lead_time_avg,length_of_stay_avg,booked_rate_avg,native_booked_rate_avg,rev_par,native_rev_par,listing_id,month_date
0,1614556800000,31,0,0.000,243.1,175,0,0,False,NaN,NaN,NaN,NaN,0.000000,0,44995702,2021-03-01
1,1617235200000,30,0,0.000,245.4,177,0,0,False,NaN,NaN,NaN,NaN,0.000000,0,44995702,2021-04-01
2,1619827200000,23,8,0.258,247.6,179,2003,1450,True,NaN,NaN,250.4,181.0,64.612903,47,44995702,2021-05-01
3,1622505600000,24,6,0.200,263.4,186,1537,1086,True,NaN,NaN,256.2,181.0,51.233333,36,44995702,2021-06-01
4,1625097600000,0,31,1.000,246.3,179,7635,5548,True,NaN,NaN,246.3,179.0,246.290323,179,44995702,2021-07-01


## 12. Monthly dataset audit

In [23]:
print("Rows:", len(monthly_df))
print("Unique listings:", monthly_df["listing_id"].nunique())
print("Min month:", monthly_df["month_date"].min())
print("Max month:", monthly_df["month_date"].max())

Rows: 306822
Unique listings: 9643
Min month: 2021-03-01 00:00:00
Max month: 2026-02-01 00:00:00


In [24]:
monthly_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 306822 entries, 0 to 306821
Data columns (total 17 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   date                    306822 non-null  int64         
 1   available_days          306822 non-null  int64         
 2   unavailable_days        306822 non-null  int64         
 3   occupancy               306822 non-null  float64       
 4   rate_avg                306822 non-null  float64       
 5   native_rate_avg         306822 non-null  int64         
 6   revenue                 306822 non-null  int64         
 7   native_revenue          306822 non-null  int64         
 8   active                  306822 non-null  bool          
 9   booking_lead_time_avg   66565 non-null   float64       
 10  length_of_stay_avg      66565 non-null   float64       
 11  booked_rate_avg         153445 non-null  float64       
 12  native_booked_rate_avg  153445 non-null  

In [25]:
monthly_df.isnull().sum().sort_values(ascending=False)

booking_lead_time_avg     240257
length_of_stay_avg        240257
native_booked_rate_avg    153377
booked_rate_avg           153377
date                           0
listing_id                     0
native_rev_par                 0
rev_par                        0
active                         0
available_days                 0
native_revenue                 0
revenue                        0
native_rate_avg                0
rate_avg                       0
occupancy                      0
unavailable_days               0
month_date                     0
dtype: int64

## 13. Clean monthly metrics

Rename the main pricing and revenue columns for clarity:

- `rate_avg` → `avg_daily_rate`
- `rev_par` → `revpar`


In [26]:
monthly_clean = monthly_df.rename(
    columns={
        "rate_avg": "avg_daily_rate",
        "rev_par": "revpar"
    }
)

monthly_clean.head()

,date,available_days,unavailable_days,occupancy,avg_daily_rate,native_rate_avg,revenue,native_revenue,active,booking_lead_time_avg,length_of_stay_avg,booked_rate_avg,native_booked_rate_avg,revpar,native_rev_par,listing_id,month_date
0,1614556800000,31,0,0.000,243.1,175,0,0,False,NaN,NaN,NaN,NaN,0.000000,0,44995702,2021-03-01
1,1617235200000,30,0,0.000,245.4,177,0,0,False,NaN,NaN,NaN,NaN,0.000000,0,44995702,2021-04-01
2,1619827200000,23,8,0.258,247.6,179,2003,1450,True,NaN,NaN,250.4,181.0,64.612903,47,44995702,2021-05-01
3,1622505600000,24,6,0.200,263.4,186,1537,1086,True,NaN,NaN,256.2,181.0,51.233333,36,44995702,2021-06-01
4,1625097600000,0,31,1.000,246.3,179,7635,5548,True,NaN,NaN,246.3,179.0,246.290323,179,44995702,2021-07-01


## 14. Export monthly clean dataset

In [27]:
monthly_clean.to_csv(
    PROCESSED_DIR / "london_monthly_metrics_clean.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "london_monthly_metrics_clean.csv")

Saved: ../data/processed/london/london_monthly_metrics_clean.csv


## 15. Final outputs summary

This notebook creates two processed London datasets:

### `london_listings_clean.csv`

Unit of analysis:

```text
1 row = 1 Airbnb listing
```

Use for:

- STR density,
- entire-home share,
- host concentration,
- professional management analysis,
- listing-level risk features,
- clustering inputs.

### `london_monthly_metrics_clean.csv`

Unit of analysis:

```text
1 row = 1 Airbnb listing + 1 month
```

Coverage:

```text
March 2021 to February 2026
```

Use for:

- monthly occupancy,
- average daily rate,
- revenue,
- RevPAR,
- historical trends,
- emerging hotspot detection.
